## Ejercicio para la toma de decisiones con base en datos 

In [ ]:
# Importaciones para dataframe
import numpy as np
import pandas as pd

#Importaciones para manejo de df
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Set de semillas para reproducibilidad
np.random.seed(42)

# Número de registros
n = 500

# Generación del dataset
df = pd.DataFrame({
    "cliente_id": range(1, n+1),
    "edad": np.random.randint(18, 65, size=n),
    "genero": np.random.choice(["Hombre", "Mujer"], size=n),
    "ciudad": np.random.choice(
        ["CDMX", "Monterrey", "Guadalajara", "Puebla", "Tijuana"],
        size=n
    ),
    "ingreso_mensual": np.random.normal(25000, 9000, size=n).astype(int),
    "visitas_mes": np.random.poisson(4, size=n),
    "compra_mes": np.random.binomial(1, 0.6, size=n),  # 0 = no compra, 1 = compra
    "monto_compra": np.abs(np.random.normal(600, 250, size=n))  # monto si compra
})

# Si el cliente no compró, monto de compra es cero
df.loc[df["compra_mes"] == 0, "monto_compra"] = 0

In [ ]:
# Indicamos número de filas y columnas
df.shape

In [ ]:
# Mostramos las primeras cinco filas del dataset
df.head()

In [ ]:
# Mostramos las últimas cinco filas del dataset
df.tail()

In [ ]:
# Mostramos 5 registros aleatorios
df.sample(5)

In [ ]:
# Estructura del DF
# Mostramos el tipo de dato de cada columna (int, float, object, datetime, etc.).
df.info()

In [ ]:
# Estadísticas descriptivas
# Generamos estadísticas de columnas numéricas (media, mediana, percentiles…).
df.describe()

In [ ]:
# Detectamos si hay nulos
df.isna().sum()

In [ ]:
#resumen automático
def resumen_eda(df):
    print("### Dimensiones ###")
    print(df.shape)
    print("\n### Tipos de Datos ###")
    print(df.dtypes)
    print("\n### Nulos ###")
    print(df.isnull().sum())
    print("\n### Estadísticas ###")
    print(df.describe())

resumen_eda(df)

## Distribución de variables numéricas

In [ ]:
# Mostramos la cantidad veces que aparece cada categoría
df['ciudad'].value_counts()

In [ ]:
df["genero"].value_counts()

In [ ]:
df["genero"].value_counts().plot(kind="bar")
plt.show()

In [ ]:
#Distribución de variables numéricas
# Generamos Histogramas automáticos para todas las columnas numéricas

df.hist(figsize=(10,6))
plt.show()

In [ ]:
# Análisis cruzado
# Promedio de ingresos por ciudad

df.groupby("ciudad")["ingreso_mensual"].mean().sort_values()


In [ ]:
# Monto promedio gastado por mes
df[df["compra_mes"] == 1]["monto_compra"].mean()

In [ ]:
# Distribución del monto de compra
sns.histplot(df["monto_compra"], kde=True)
plt.title("Distribución del Monto de Compra")
plt.show()

In [ ]:
# Relación entre visitas y compras
# Visitar más se relaciona con gastar más?
sns.scatterplot(x="visitas_mes", y="monto_compra", data=df)
plt.show()

In [ ]:
sns.scatterplot(x="edad", y="monto_compra", data=df)
plt.title("Relación Edad vs Monto de Compra")
plt.show()

In [ ]:
# Graficamos la forma de la distribución (sesgada, normal, picos…).

sns.histplot(df['ingreso_mensual'], kde=True)
plt.title("Distribución de Ingresos")
plt.show()

In [ ]:
#Matriz de correlación

#Nos quedamos solo con variables numéricas
numeric_df = df.select_dtypes(include=[np.number])

corr = numeric_df.corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='Blues')
plt.title("Matriz de Correlación")
plt.show()


In [ ]:
df.to_csv("clientes_venta.csv", index=False)

Preguntas de caso:

¿Qué factores predicen mejor que un cliente realice una compra?
¿Hay ciudades con mejor rendimiento comercial?
¿Qué insights se detectan de la relación edad–gasto?
¿Qué decisión propondrías al área comercial basada en los datos?

Ejercicio:

Identificación de un hallazgo / generar Insights
Interpretación de hallazgo
Integrara una recomendación asociada
¿Qué decisión de negocio tomarían?

In [ ]:
# 1. IDENTIFICACIÓN DE HALLAZGOS Y GENERACIÓN DE INSIGHTS

# A) Tasa de conversión y ticket promedio por ciudad
resumen_ciudad = df.groupby('ciudad').agg(
    tasa_conversion=('compra_mes', 'mean'),
    ticket_promedio_compradores=('monto_compra', lambda x: x[x>0].mean()) # Promedio excluyendo los ceros
).reset_index().sort_values('ticket_promedio_compradores', ascending=False)

print("--- Rendimiento Comercial por Ciudad ---")
print(resumen_ciudad)

In [ ]:
# B) Correlaciones clave
corr_visitas_compra = df['visitas_mes'].corr(df['compra_mes'])
corr_ingreso_monto = df['ingreso_mensual'].corr(df['monto_compra'])

print(f"\n--- Correlaciones Clave ---")
print(f"Visitas vs Probabilidad de Compra: {corr_visitas_compra:.3f}")
print(f"Ingreso Mensual vs Monto Gastado: {corr_ingreso_monto:.3f}")